# DEEPX Tutorial 03 - AI Project Workflow with DEEPX NPU

This third tutorial demonstrates the full end-to-end workflow for AI model deployment on DEEPX hardware.

We will train a forklift and worker detection model, utilize the DX-Compiler tool for conversion to the DXNN format, and execute the final AI application on a DEEPX NPU. This process will provide a comprehensive picture of the DEEPX NPU development pipeline. 

## Hands-on Project Overview

- **Detection classes**: Forklift, Worker
- **Base AI model**: YOLOv7
- **Dataset**: 1448 images of Forklift & Worker from [Kaggle](https://www.kaggle.com/datasets/hakantaskiner/personforklift-dataset/data)
- **Train**: Need NVIDIA GPU with more than 24G GRAM (Training requires heavy GPUs, but Deployment only needs DEEPX NPU)
- **Inference NPU**: `DX-M1`
- **AI application**: Modify and reuse the yolo demo of the DX-APP
- **Expected output**:
<img src="assets/detection-goal.jpg" style="max-width: 1200px;">

## AI Workflow Overview

This diagram explains the common workflow of an AI project.

We define the goal, collect and label data, and train the model.
DX-Compiler helps make the model faster and lighter (INT8) for DX NPU.
The final step is deploying the model to the DEEPX NPU using DX-APP or DX-STREAM.

Each step builds toward real-world AI solutions, such as worker and forklift detection.

  <img src="assets/workflow2.jpg" style="max-width: 1200px;">

## 1. AI Workflow - Model Selection based on the use case

To start an AI project, we need to select an AI model that fits the use case.

In this tutorial, our goal is to detect forklifts and workers.
We will use YOLOv7, a well-known model for object detection.

- Choose YOLOv7 to detect Forklift & Worker
- For more details of YOLOv7: 👉 [link](https://docs.ultralytics.com/models/yolov7/)
- How to use YOLOv7: 👉 [link](https://github.com/WongKinYiu/yolov7)

## 2. AI Workflow - Data Preparation & Annotation

Download the forklift-person labeled dataset from Kaggle:
 - Reference: [Kaggle link](https://www.kaggle.com/datasets/hakantaskiner/personforklift-dataset)

## 3. AI Workflow - Train

To train the model efficiently, you should use a GPU that has 24GB or more graphic memory.

 - How to train YOLOv7: 👉 [Link](https://colab.research.google.com/drive/1dAdjJuhXqFM_Qcd0QqAn7_AGx7abA5aX?usp=sharing)

## 4. AI Workflow - Optimization with DX-Compiler

In [ ]:
# Load all SDK paths from dx-tutorials/config.json.
import os
root_path = os.environ.get("ROOT_PATH")
if not root_path:
    raise EnvironmentError("ROOT_PATH is not set. Start JupyterLab with ./run-jupyter-lab.sh")
%run "$root_path/tutorial_paths.py"
print_tutorial_paths()
%cd $DX_COMPILER_DIR/dx_com

Let's compile your pre-trained AI model to DXNN format.

Overall processes are:
1. Get a pre-trained model on pytorch framework
2. Convert it to ONNX format
3. Compile ONNX to DXNN (for more details of DX-Compiler, refer to the user guide 👉  [here](https://developer.deepx.ai/download/?id=581)
                                                                                             
> Note: To download the User Guide, you must log in to https://developer.deepx.ai/ first.

<img src="assets/dx-com-workflow.jpg" style="max-width: 1200px;">

The source structure of DX-Compiler is organized as follows:
```bash
dx_com
 ├── calibration_dataset   # Dataset used to optimize model accuracy
 └── sample_models         # Sample configuration file and ONNX files 
```

In [ ]:
!tree -L 1
!tree sample_models

### 4.1 Prepare the exported ONNX model and YOLOv7 DX-Compiler configuration

The custom YOLOv7 ONNX model is downloaded directly. This tutorial also includes a copy of the YOLOv7 Q-Lite JSON configuration so the Notebook can run from top to bottom without assuming a browser download location.

You can inspect the original configuration in the DEEPX Model Zoo: Object Detection >> YOLOv7 >> Q-Lite JSON download 👉 [modelzoo](https://developer.deepx.ai/modelzoo/)

<img src="assets/sc-modelzoo-yolov7.png" style="max-width: 1200px;">

  
  `Note`: The DEEPX Model Zoo provides AI models and compiler configurations verified by DEEPX.

In [ ]:
# Download the custom YOLOv7 ONNX model only when it is not already present.
!wget -nc "https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/yolov7-forklift-person.onnx"

# Start from the tutorial's bundled Model Zoo configuration.
!cp "$TUTORIAL_ROOT/notebooks/T03-E2E-AI-Workflow/assets/YOLOV7-2.json" ./yolov7-forklift-person.json

### 4.2 Modify YOLOv7 json file for your custom env

In [ ]:
# Check the original json file downloaded from DX model zoo
!cat yolov7-forklift-person.json

In [ ]:
# Change the dataset_path
# "dataset_path": "/mnt/datasets/COCO/official/images/val2017" -> "./calibration_dataset"

import json

file_name = 'yolov7-forklift-person.json'

# 1. Read JSON file
with open(file_name, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Find and replace dictionary values by key
data['default_loader']['dataset_path'] = "./calibration_dataset"

# 3. Save the revised json file
with open(file_name, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4)

print(f"'{file_name}' update done")

### 4.3 Compile ONNX to DXNN by referencing json configuration

DX-Compiler uses a dedicated Python virtual environment created during SDK installation. JupyterLab runs in the separate `dx-tutorials` environment, so `dxcom` is not normally available on the Notebook's `PATH`.

The next code cell derives the DX-Compiler paths from `config.json`, activates the DX-Compiler environment, moves to the `dx_com` workspace, and runs `dxcom` in the same shell. Each `!` code cell starts a new shell, so the activation does not persist across cells. For this reason, every Notebook cell that runs `dxcom` repeats the activation command.

Outside this tutorial, open a terminal and activate the environment once:

```bash
cd <DX_ALL_SUITE_DIR>/dx-compiler
source venv-dx-compiler-local/bin/activate
cd dx_com
dxcom -h
```

The activated environment remains active in that terminal until you run `deactivate` or close the terminal.

In [ ]:
DX_COM_DIR = DX_COMPILER_DIR / "dx_com"
DX_COMPILER_VENV = DX_COMPILER_DIR / "venv-dx-compiler-local"

!source "{DX_COMPILER_VENV}/bin/activate" && cd "{DX_COM_DIR}" && dxcom -h

#### 4.3.1 **(Option A)** Manual Compilation 
If you want to skip compile process, just jump to the "4.3.2 (Option B)" and download "yolov7-forklift-person.dxnn" directly.
> **Note**: The following compile command will take **more than 10 mins** depending on your system performance.

In [ ]:
!source "{DX_COMPILER_VENV}/bin/activate" && \
 cd "{DX_COM_DIR}" && \
 dxcom -m yolov7-forklift-person.onnx \
       -c yolov7-forklift-person.json \
       -o output \
       --gen_log

# Move the compiled dxnn file to the dx_app path
!mv output/yolov7-forklift-person.dxnn $DX_APP_DIR/

#### 4.3.2 **(Option B)** Download Pre-compiled Model
If you want to skip compile process, just download the compiled DXNN file from the following link after uncommenting:

In [ ]:
#!wget "cs.deepx.ai/_deepx_fae_archive/dx-tutorials/yolov7-forklift-person.dxnn"

#!mv yolov7-forklift-person.dxnn $DX_APP_DIR/

## 5. AI Workflow - Deployment on DEEPX NPU

In [ ]:
# Move to the configured DX-APP directory.
%cd $DX_APP_DIR

### 5.1 Configure DX-APP for the Custom YOLOv7 Model

The current DX-APP does not require a C++ source patch for a custom class count. The YOLOv7 factory reads `num_classes` and `class_names` from a runtime configuration file, and the postprocessor uses those values when it decodes detections.

Do not confuse the two JSON configuration files used in this tutorial:

| Configuration | Used by | Purpose |
|---|---|---|
| `yolov7-forklift-person.json` | `dxcom -c` | Describes model compilation, input shape, preprocessing, and calibration data. |
| `yolov7-forklift-runtime.json` | DX-APP `--config` | Describes runtime post-processing thresholds, the class count, and display labels. |

The compiled model has a decoded output shape of `[1, 25200, 7]`. Each row contains four box values, one objectness value, and two class scores: `4 + 1 + 2 = 7`. The raw detection heads contain 21 channels because each head uses three anchors: `3 × (5 + 2) = 21`.

In [ ]:
%%writefile yolov7-forklift-runtime.json
{
    "obj_threshold": 0.25,
    "score_threshold": 0.25,
    "nms_threshold": 0.45,
    "num_classes": 2,
    "class_names": [
        "Forklift",
        "Worker"
    ]
}

#### Runtime Configuration Fields

| Field | Description |
|---|---|
| `obj_threshold` | Rejects candidates whose objectness score is below this value. |
| `score_threshold` | Rejects detections whose final class confidence is below this value. For YOLOv7, the final confidence is based on objectness and the class score. |
| `nms_threshold` | IoU threshold used by Non-Maximum Suppression to remove overlapping detections. A lower value removes overlapping boxes more aggressively. |
| `num_classes` | Number of classes produced by the model. This model produces two classes, so the value must be `2`. |
| `class_names` | Labels displayed for each class ID. The array order must exactly match the class order used during training. |

For this model, class ID `0` is `Forklift` and class ID `1` is `Worker`. Changing only the label text does not change model behavior; it only changes how the detected class IDs are displayed. A mismatch between `num_classes`, `class_names`, and the model output can produce incorrect decoding or incorrect labels.

In [ ]:
# Review the DX-APP runtime configuration.
!cat yolov7-forklift-runtime.json

### 5.2 Verify the YOLOv7 Executable

The runtime configuration is loaded with `--config` before DX-APP creates the YOLOv7 postprocessor. Because no C++ source is changed, rebuilding DX-APP is not required when `bin/yolov7_async` already exists.

If the executable is missing, build only this target in a separate terminal:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./build.sh --target yolov7_async
```

In [ ]:
!./bin/yolov7_async -h

### 5.3 Run the Custom Model with an Image

Pass both the custom DXNN model and its runtime configuration to the standard YOLOv7 executable. The shared DX-APP source remains unchanged.

In [ ]:
!cp $TUTORIAL_ROOT/notebooks/T03-E2E-AI-Workflow/assets/forklift-worker.png ./

In [ ]:
!./bin/yolov7_async \
    -m yolov7-forklift-person.dxnn \
    --config yolov7-forklift-runtime.json \
    -i forklift-worker.png

### 5.4 Run the Custom Model with a Video

In [ ]:
!wget "https://cs.deepx.ai/_deepx_fae_archive/dx-tutorials/forklift-worker.mp4"

In [ ]:
!./bin/yolov7_async \
    -m yolov7-forklift-person.dxnn \
    --config yolov7-forklift-runtime.json \
    -v forklift-worker.mp4